# 🏦 Register an Agent in Microsoft Foundry (so it shows in the Agents list)

## Why your `Agent` didn't appear in Foundry

The other notebooks build an agent like this:

```python
Agent(client=FoundryChatClient(...), name="CreditAnalyst", instructions=...)
```

That is a **client-side** agent — it lives only in this Python process and just *calls* your model deployment. Nothing is created on the server, so the Foundry portal's **Agents** list has nothing to show. The same is true for `AzureChatClient` / `AzureOpenAIChatClient` (direct Azure OpenAI chat).

| | Client-side (`FoundryChatClient` / `AzureChatClient`) | **Registered** (this notebook) |
|---|---|---|
| Lives | In your process (memory) | Server-side resource in your project |
| Shows in portal → **Agents** | ❌ No | ✅ Yes |
| Lifetime | Gone when the cell ends | Persists until you delete it |
| How to run it | `agent.run(...)` | `FoundryAgent(agent_name=...)` or the Responses API |

## What this notebook does

1. Write the agent the way you already do (`Agent` + `FoundryChatClient`).
2. Bridge it to a Foundry **`PromptAgentDefinition`** with `to_prompt_agent()`.
3. **Persist** it with `AIProjectClient.agents.create_version()` → now it appears in Foundry.
4. Run the *registered* agent with the Agent Framework's `FoundryAgent`.
5. Confirm it's listed, then clean up.

> ⚠️ Educational demo only — the credit logic is simplified and not a real lending decision.

## Prerequisites
- `az login` (uses `AzureCliCredential`)
- Repo `.env` with a Foundry project endpoint + model deployment
- The repository `.venv` kernel selected


## 1️⃣ Imports and environment

In [ ]:
# Copyright (c) Microsoft. All rights reserved.
import asyncio
import os
import sys
from importlib.metadata import version
from pathlib import Path

from agent_framework import Agent
from agent_framework.foundry import FoundryAgent, FoundryChatClient, to_prompt_agent
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import PromptAgentDefinition
from azure.identity import AzureCliCredential
from dotenv import load_dotenv

assert version("agent-framework-core") == "1.17.0", "Select the pinned project kernel."
repo_root = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "requirements.in").is_file())
assert Path(sys.executable).resolve() == (repo_root / ".venv/Scripts/python.exe").resolve(), (
    "Select the repository .venv kernel."
)
load_dotenv(repo_root / ".env", override=False)

PROJECT_ENDPOINT = (
    os.getenv("FOUNDRY_PROJECT_ENDPOINT")
    or os.getenv("AI_FOUNDRY_PROJECT_ENDPOINT")
    or os.getenv("AZURE_AI_PROJECT_ENDPOINT")
)
MODEL_DEPLOYMENT = os.getenv("FOUNDRY_MODEL") or os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")
AGENT_NAME = "credit-analyst-agent"  # the name that will show up in Foundry

if not PROJECT_ENDPOINT or not MODEL_DEPLOYMENT:
    raise ValueError("Set a Foundry project endpoint and model deployment name in the repo .env.")
print("✅ Imports loaded; project endpoint and model configured (values hidden)")

## 2️⃣ Write the agent the way you already do — then bridge it to Foundry

`to_prompt_agent()` takes the **same** client-side `Agent` (bound to a `FoundryChatClient`) and turns it into a Foundry `PromptAgentDefinition`. The model deployment is lifted automatically from the client. This is the bridge from "code-only" to "registered".

In [ ]:
CREDIT_ANALYST_INSTRUCTIONS = (
    "You are a Credit Analyst at a retail bank reviewing credit card applications.\n"
    "For each application: evaluate income and employment stability, assess the "
    "debt-to-income ratio, review the credit score and history, flag any risk factors, "
    "and give a concise preliminary creditworthiness assessment.\n"
    "Always end with a one-line note that this is a demonstration, not a real lending decision."
)


async def build_prompt_agent_definition() -> PromptAgentDefinition:
    """Build the client-side Agent you already write, then convert it for Foundry."""
    # On its own, this Agent is NOT registered in Foundry — it only calls the model.
    chat_client = FoundryChatClient(
        project_endpoint=PROJECT_ENDPOINT,
        model=MODEL_DEPLOYMENT,
        credential=AzureCliCredential(),
    )
    try:
        analyst = Agent(
            client=chat_client,
            name="CreditAnalyst",
            instructions=CREDIT_ANALYST_INSTRUCTIONS,
        )
        # Bridge: agent-framework Agent -> Foundry PromptAgentDefinition (model lifted from client).
        return to_prompt_agent(analyst)
    finally:
        await chat_client.client.close()
        await chat_client.project_client.close()


definition = await build_prompt_agent_definition()
print(f"✅ Built a PromptAgentDefinition (model: {definition.model}) — not registered yet")

## 3️⃣ Persist it to Foundry — this is what makes it appear

`create_version()` registers a **server-side** agent version in your project. After this cell runs, open **Microsoft Foundry → your project → Agents** and you'll see `credit-analyst-agent`.

In [ ]:
# Reused across the notebook; closed in the final cleanup cell.
credential = AzureCliCredential()
project_client = AIProjectClient(endpoint=PROJECT_ENDPOINT, credential=credential)

registered = project_client.agents.create_version(
    agent_name=AGENT_NAME,
    description="Credit analyst that reviews credit card applications (demo).",
    definition=definition,
)
print(f"✅ Registered in Foundry: {registered.name} (version {registered.version})")
print("   Open Microsoft Foundry → your project → Agents to see it in the list.")

## 4️⃣ Run the *registered* agent with `FoundryAgent`

`FoundryAgent` connects to the agent **by name** in your project — the same one now visible in the portal — and runs it with Agent Framework streaming.

In [ ]:
CREDIT_APPLICATION = """
CREDIT CARD APPLICATION
Applicant: Sarah Johnson | Age: 32 | Homeowner (5 yrs)
Annual Income: $85,000 | Employment: Marketing Manager (4 yrs)
Monthly mortgage: $1,800 | Auto loan: $15,000 ($350/mo)
Existing cards: 2 (limit $12,000, 25% utilization) | Credit score: 745
Requested limit: $10,000
"""


async def review_application() -> None:
    """Stream a review from the Foundry-registered agent."""
    async with FoundryAgent(
        agent_name=AGENT_NAME,
        agent_version=registered.version,
        project_endpoint=PROJECT_ENDPOINT,
        credential=credential,
        timeout=90,
    ) as agent:
        print("💳 Credit application review (streamed from the Foundry-registered agent):\n")
        async with asyncio.timeout(120):
            async for update in agent.run(CREDIT_APPLICATION, stream=True):
                if update.text:
                    print(update.text, end="", flush=True)
        print("\n\n✅ That ran the agent registered in your Foundry project.")


await review_application()

## 5️⃣ Confirm it's registered

In [ ]:
versions = list(project_client.agents.list_versions(agent_name=AGENT_NAME))
print(f"'{AGENT_NAME}' has {len(versions)} version(s) registered in Foundry:")
for v in versions:
    print(f"  - version {v.version}")

## 6️⃣ Cleanup

Delete only the version this notebook created, then close the SDK clients. Skip this cell if you want to keep the agent visible in the portal to explore.

In [ ]:
try:
    project_client.agents.delete_version(agent_name=AGENT_NAME, agent_version=registered.version)
    print(f"🧹 Deleted {AGENT_NAME} version {registered.version}")
except Exception as exc:
    print(f"⚠️ Cleanup issue (delete manually in the portal if needed): {exc}")
finally:
    project_client.close()
    credential.close()
    print("✅ Clients closed")

## 📝 Key takeaways

| If you use… | Then… |
|---|---|
| `FoundryChatClient` / `AzureChatClient` + `Agent` | Runs from your code; **not** in the Foundry Agents list |
| `to_prompt_agent()` + `AIProjectClient.agents.create_version()` | **Registers** a server-side agent → **appears in Foundry** |
| `FoundryAgent(agent_name=...)` | Connects to and runs the **registered** agent |

- Registration is **versioned** — re-running `create_version` adds a new version under the same name.
- Delete with `project_client.agents.delete_version(...)` so demo agents don't pile up.
- Prefer no code at all? Create the agent directly in the Foundry portal, or deploy a **hosted agent** (see `hosted-agents/`) — both also show in the Agents list.
